In [15]:
# Ячейка для отладки загрузки аудио и определения T

import soundfile as sf
import librosa
from pathlib import Path
import math
import traceback

# --- Параметры (скопируй из конфига или задай явно) ---
AUDIO_CONFIG = { "sample_rate": 8000, "hop_length": 96, "n_fft": 512 }
PATHS_CONFIG = { "data_dir": "./", "audio_folder_name": "morse_dataset/morse_dataset" }
# -------------------------------------------------------

# --- Укажи ID реального существующего файла ---
TEST_FILE_ID = "1" # Или любой другой ID, например '0a1d2f'
# -------------------------------------------

data_dir = Path(PATHS_CONFIG["data_dir"])
audio_folder_path = data_dir / PATHS_CONFIG["audio_folder_name"]
test_audio_path = audio_folder_path / f"{TEST_FILE_ID}.opus"

print(f"Проверяем файл: {test_audio_path.resolve()}")

if not test_audio_path.exists():
    print("!!! ОШИБКА: Тестовый файл не найден!")
else:
    print("Тестовый файл найден. Попытка получить информацию...")
    T = None
    try:
        # --- Попытка 1: Использовать sf.info ---
        print("\n--- Попытка 1: sf.info ---")
        info = sf.info(str(test_audio_path))
        print(f"sf.info результат: {info}")

        num_samples = info.frames
        sr = info.samplerate
        target_sr = AUDIO_CONFIG["sample_rate"]
        hop_length = AUDIO_CONFIG["hop_length"]

        if sr != target_sr:
            num_samples = int(num_samples * (target_sr / sr))
            print(f"Оценочное кол-во сэмплов после ресемплинга до {target_sr} Hz: {num_samples}")
        else:
            print(f"Количество сэмплов (без ресемплинга): {num_samples}")


        if num_samples > 0 and hop_length > 0:
            T = math.floor(num_samples / hop_length) + 1
            print(f"Рассчитанное T: {T}")
        else:
            print("Ошибка: num_samples или hop_length некорректны.")

    except Exception as e1:
        print(f"❌ Ошибка при использовании sf.info: {e1}")
        traceback.print_exc(limit=2)

        # --- Попытка 2: Использовать librosa.get_duration ---
        # (Может использовать soundfile или audioread под капотом)
        if T is None: # Пробуем второй метод, если первый не удался
            print("\n--- Попытка 2: librosa.get_duration ---")
            try:
                duration = librosa.get_duration(path=test_audio_path)
                print(f"librosa.get_duration результат: {duration:.4f} секунд")

                target_sr = AUDIO_CONFIG["sample_rate"]
                hop_length = AUDIO_CONFIG["hop_length"]
                num_samples = int(duration * target_sr) # Оцениваем кол-во сэмплов

                if num_samples > 0 and hop_length > 0:
                    T = math.floor(num_samples / hop_length) + 1
                    print(f"Рассчитанное T: {T}")
                else:
                    print("Ошибка: num_samples или hop_length некорректны.")

            except Exception as e2:
                print(f"❌ Ошибка при использовании librosa.get_duration: {e2}")
                traceback.print_exc(limit=2)

        # --- Попытка 3: Загрузить данные и посчитать ---
        if T is None: # Пробуем третий метод, если первые два не удались
             print("\n--- Попытка 3: sf.read + расчет ---")
             try:
                 y, sr = sf.read(test_audio_path, dtype='float32')
                 print(f"sf.read успешно загрузил данные: shape={y.shape}, sr={sr}")
                 target_sr = AUDIO_CONFIG["sample_rate"]
                 hop_length = AUDIO_CONFIG["hop_length"]

                 # Ресемплинг, если нужно
                 if sr != target_sr:
                     y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
                     print(f"Данные ресемплированы до {target_sr} Hz, новая shape={y.shape}")

                 num_samples = len(y)
                 if num_samples > 0 and hop_length > 0:
                     T = math.floor(num_samples / hop_length) + 1
                     print(f"Рассчитанное T: {T}")
                 else:
                     print("Ошибка: num_samples или hop_length некорректны.")

             except Exception as e3:
                 print(f"❌ Ошибка при использовании sf.read: {e3}")
                 traceback.print_exc(limit=2)

    if T is not None and T > 0:
        print(f"\n✅ Удалось определить T = {T} для файла {TEST_FILE_ID}.opus")
    else:
        print(f"\n❌ Не удалось определить корректное T для файла {TEST_FILE_ID}.opus")

Проверяем файл: C:\Users\vasja\OneDrive\Рабочий стол\Morse_mel\MorseAudioDecoder\morse_dataset\morse_dataset\1.opus
Тестовый файл найден. Попытка получить информацию...

--- Попытка 1: sf.info ---
sf.info результат: morse_dataset\morse_dataset\1.opus
samplerate: 8000 Hz
channels: 1
duration: 8.000 s
format: OGG (OGG Container format) [OGG]
subtype: Opus [OPUS]
Количество сэмплов (без ресемплинга): 64000
Рассчитанное T: 667

✅ Удалось определить T = 667 для файла 1.opus


In [16]:
# generate_generic_perlin_maps.py (v3 - С продолжением нумерации)
import numpy as np
from perlin_noise import PerlinNoise
from pathlib import Path
from tqdm.notebook import tqdm # или from tqdm import tqdm
import random
import json
import traceback
import os
import math
import re # Добавляем модуль для регулярных выражений
from typing import Optional, Dict, Tuple

# --- Параметры Генерации ---
F_FIXED = 257
T_FIXED = 667
# --- !!! ИЗМЕНЕНО: Теперь это СКОЛЬКО ВСЕГО карт мы хотим иметь !!! ---
N_MAPS_TARGET = 2000 # Например, хотим в итоге 2000 пар карт
# --------------------------------------------------------------------
RANDOM_SEED = 42 # Сид для случайности генерации, не для старта нумерации
OUTPUT_DIR = "./generic_perlin_maps"

PERLIN_PARAMS = {
    "octaves_min": 1, "octaves_max": 1,
    "scale_pow_min": 3, "scale_pow_max": 8,
}
# ---------------------------

# --- Установка Seed ---
def set_seed(seed): random.seed(seed); np.random.seed(seed)
set_seed(RANDOM_SEED)
# ----------------------

# --- Функция генерации сырого Перлина [-1, 1] (та же) ---
def _generate_raw_perlin(shape: tuple, octaves: int, seed: int, scale: float) -> Optional[np.ndarray]:
    if not all(s > 0 for s in shape) or scale <= 0 or octaves < 1: return None
    try:
        noise_gen = PerlinNoise(octaves=octaves, seed=seed)
        height, width = shape; x_coords = np.arange(width)/scale; y_coords = np.arange(height)/scale
        xv, yv = np.meshgrid(x_coords, y_coords)
        raw_noise = np.array([[noise_gen([xv[i,j], yv[i,j]]) for j in range(width)] for i in range(height)])
        max_abs_val = np.max(np.abs(raw_noise)); norm_noise = raw_noise / max_abs_val if max_abs_val > 1e-6 else raw_noise
        if np.isnan(norm_noise).any(): print(f"!!! Ошибка: NaN в сыром шуме (seed={seed}, scale={scale:.2f})"); return None
        return norm_noise.astype(np.float32)
    except ValueError as ve: print(f"!!! ValueError в PerlinNoise (seed={seed}, scale={scale:.2f}): {ve}"); return None
    except Exception as e: print(f"!!! Неизвестная ошибка _generate_raw_perlin (seed={seed}, scale={scale:.2f}): {e}"); return None
# -----------------------------------------------------------------------

# --- Основной Цикл Генерации Карт ---
maps_dir = Path(OUTPUT_DIR)
maps_dir.mkdir(parents=True, exist_ok=True)
print(f"Проверка существующих карт в: {maps_dir.resolve()}")

# --- !!! Определение начального индекса !!! ---
start_index = 0
existing_files = list(maps_dir.glob("offset_map_*.npy")) # Ищем файлы смещения
if existing_files:
    max_existing_index = -1
    # Используем регулярное выражение для извлечения номера
    pattern = re.compile(r"offset_map_(\d+)\.npy")
    for f in existing_files:
        match = pattern.match(f.name)
        if match:
            try:
                index = int(match.group(1))
                max_existing_index = max(max_existing_index, index)
            except ValueError:
                pass # Игнорируем файлы с нечисловыми индексами
    if max_existing_index >= 0:
        start_index = max_existing_index + 1
print(f"Найдено {len(existing_files)} существующих карт смещения.")
print(f"Генерация начнется с индекса: {start_index}")
# ---------------------------------------------

# --- Рассчитываем, сколько еще нужно сгенерировать ---
n_maps_to_generate_now = max(0, N_MAPS_TARGET - start_index)
if n_maps_to_generate_now == 0:
    print(f"Целевое количество карт ({N_MAPS_TARGET}) уже достигнуто или превышено. Генерация не требуется.")
else:
    print(f"Цель: {N_MAPS_TARGET} пар карт. Нужно сгенерировать еще: {n_maps_to_generate_now}")
    print(f"Фиксированный размер карт: ({F_FIXED}, {T_FIXED})")

    fixed_shape = (F_FIXED, T_FIXED)
    generated_pairs_count_session = 0 # Счетчик для текущей сессии
    error_count_session = 0

    # --- Цикл генерации от start_index до N_MAPS_TARGET ---
    for i in tqdm(range(start_index, N_MAPS_TARGET), desc="Генерация карт"):
        offset_map = None; multi_map = None
        offset_saved = False; multi_saved = False

        # --- Генерация Карты Смещения ---
        try:
            for attempt in range(5):
                offset_octaves=random.randint(PERLIN_PARAMS["octaves_min"], PERLIN_PARAMS["octaves_max"]); offset_r1=random.randint(PERLIN_PARAMS["scale_pow_min"], PERLIN_PARAMS["scale_pow_max"]); offset_r2=random.uniform(1.0, 2.0); offset_scale=(2**offset_r1)*offset_r2; offset_seed=random.randint(1, 1_000_000)+i*10+attempt
                offset_map = _generate_raw_perlin(fixed_shape, offset_octaves, offset_seed, offset_scale)
                if offset_map is not None: break
            if offset_map is None: raise RuntimeError("Не удалось сгенерировать карту смещения")
            save_path = maps_dir / f"offset_map_{i:04d}.npy"
            np.save(save_path, offset_map, allow_pickle=False); offset_saved = True
        except Exception as e_offset: print(f"!!! Ошибка генерации/сохранения offset_map_{i:04d}: {e_offset}"); error_count_session += 1

        # --- Генерация Карты Множителя ---
        try:
            for attempt in range(5):
                multi_octaves=random.randint(PERLIN_PARAMS["octaves_min"], PERLIN_PARAMS["octaves_max"]); multi_r1=random.randint(PERLIN_PARAMS["scale_pow_min"], PERLIN_PARAMS["scale_pow_max"]); multi_r2=random.uniform(1.0, 2.0); multi_scale=(2**multi_r1)*multi_r2; multi_seed=random.randint(1_000_001, 2_000_000)+i*10+attempt
                multi_map = _generate_raw_perlin(fixed_shape, multi_octaves, multi_seed, multi_scale)
                if multi_map is not None: break
            if multi_map is None: raise RuntimeError("Не удалось сгенерировать карту множителя")
            save_path = maps_dir / f"multi_map_{i:04d}.npy"
            np.save(save_path, multi_map, allow_pickle=False); multi_saved = True
        except Exception as e_multi: print(f"!!! Ошибка генерации/сохранения multi_map_{i:04d}: {e_multi}"); error_count_session += 1

        if offset_saved and multi_saved:
            generated_pairs_count_session += 1
    # --- Конец цикла генерации ---

    print(f"\nГенерация карт (сессия) завершена.")
    print(f"Успешно сгенерировано новых пар карт: {generated_pairs_count_session}")
    print(f"Всего ошибок генерации/сохранения (сессия): {error_count_session}")

# --- Финальный подсчет общего количества карт ---
final_offset_files = list(maps_dir.glob("offset_map_*.npy"))
final_multi_files = list(maps_dir.glob("multi_map_*.npy"))
print(f"\nИтого в папке {maps_dir.resolve()}:")
print(f"  Карт смещения: {len(final_offset_files)}")
print(f"  Карт множителя: {len(final_multi_files)}")

Проверка существующих карт в: C:\Users\vasja\OneDrive\Рабочий стол\Morse_mel\MorseAudioDecoder\generic_perlin_maps
Найдено 1374 существующих карт смещения.
Генерация начнется с индекса: 1374
Цель: 2000 пар карт. Нужно сгенерировать еще: 626
Фиксированный размер карт: (257, 667)


Генерация карт:   0%|          | 0/626 [00:00<?, ?it/s]

KeyboardInterrupt: 